In [6]:
import json
import sys
from pathlib import Path
import re

# 将当前目录加入系统路径
sys.path.append(str(Path.cwd()))

from config import OLLAMA_CHAT_MODEL
from utils import ollama_chat
from prompts import EXTRACT_LOCAL_PROMPT, fill_prompt

print(f"当前模型：{OLLAMA_CHAT_MODEL}")

当前模型：qwen3.5:9b


In [7]:
# 方式一：从文件加载（如果你保存了）
mind_map_file = "data/mind_map_sample.json"
if Path(mind_map_file).exists():
    with open(mind_map_file, "r", encoding="utf-8") as f:
        mind_map_tree = json.load(f)
    print("已从文件加载思维导图树。")
else:
    mind_map_tree = None
    print("未找到思维导图文件，将使用手动定义的叶子问题列表。")

# 提取叶子问题列表（用于后续检索）
def collect_leaf_questions(nodes):
    leafs = []
    for node in nodes:
        if not node.get("children"):
            leafs.append(node["sub_question"])
        else:
            leafs.extend(collect_leaf_questions(node["children"]))
    return leafs

if mind_map_tree:
    leaf_questions = collect_leaf_questions(mind_map_tree)
else:
    # 手动定义一个测试问题列表（你可以替换成你自己的）
    leaf_questions = [
    ]

print("\n叶子问题列表：")
for i, q in enumerate(leaf_questions, 1):
    print(f"{i}. {q}")

已从文件加载思维导图树。

叶子问题列表：
1. 陈氏太极拳的动作要素有哪些？
2. 陈氏太极拳的代表传承人有哪些？


In [8]:
def extract_local_keys(question_list):
    prompt = fill_prompt(EXTRACT_LOCAL_PROMPT, mind_map=json.dumps(question_list, ensure_ascii=False))
    print("正在调用 LLM 提取实体...")
    response = ollama_chat(prompt, temperature=0.1, format_json=False)
    
    print(f"[调试] 原始响应长度：{len(response)} 字符")
    print(f"[调试] 原始响应前300字符：{repr(response[:300])}")
    
    json_match = re.search(r'\{.*\}', response, re.DOTALL)
    if not json_match:
        print("未找到 JSON 对象，返回空结果")
        return {"entities": []}
    
    try:
        extracted = json.loads(json_match.group(0))
        result = {"entities": extracted.get("entities", [])}
        return result
    except Exception as e:
        print(f"解析 JSON 失败：{e}")
        return {"entities": []}

In [9]:
keys = extract_local_keys(leaf_questions)

print("\n===== 提取结果 =====")
print(f"实体 ({len(keys['entities'])} 个):")
for e in keys['entities']:
    print(f"  - {e}")

正在调用 LLM 提取实体...
[调试] 原始响应长度：38 字符
[调试] 原始响应前300字符：'{"entities": ["陈氏太极拳", "动作要素", "传承人"]}'

===== 提取结果 =====
实体 (3 个):
  - 陈氏太极拳
  - 动作要素
  - 传承人


In [10]:
output_file = "data/extracted_keys.json"
Path("data").mkdir(exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(keys, f, ensure_ascii=False, indent=2)

print(f"提取的 Key 已保存至：{output_file}")

提取的 Key 已保存至：data/extracted_keys.json
